# Pipeline 2 — Modelo predictivo y visualización
**Tarea:** aprendizaje supervisado (regresión). **Objetivo:** predecir minutos de viaje por acarreo a partir de distancia (km), carga (t) y ley del mineral (%). Los 240 registros son **sintéticos**; la ley se incluye como atributo de ejercicio, no como un determinante físico establecido del viaje. No se entrena con datos del caso de cuatro yacimientos, que resultan insuficientes.

## 1. Carga y exploración de datos
Se utiliza una muestra reproducible con semilla 42. Se permite ejecutar el notebook sin clonar: usa los datos de GitHub si se reemplaza `USUARIO`, o genera el mismo esquema de datos localmente.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

local = Path('../data/operaciones_sinteticas.csv')
url = 'https://raw.githubusercontent.com/josue456cris/Mineria-Pipeline/main/data/operaciones_sinteticas.csv'
try:
    df = pd.read_csv(local) if local.exists() else pd.read_csv(url)
except Exception:
    rng = np.random.default_rng(42)
    n = 240
    dist = rng.uniform(5,30,n); carga = rng.uniform(80,180,n); ley = rng.uniform(.3,1.3,n)
    tiempo = 8 + 2.35*dist + .055*carga - .8*ley + rng.normal(0,3,n)
    df = pd.DataFrame({'distancia_km':dist,'ley_cobre_pct':ley,'carga_ton':carga,'tiempo_viaje_min':tiempo.clip(min=1)})
print('Filas:', len(df)); display(df.head()); display(df.describe())

## 2. Preparación y división
Separamos un 20 % para prueba antes de ajustar el preprocesamiento. Así se evita fuga de información.

In [ ]:
X = df[['distancia_km', 'carga_ton', 'ley_cobre_pct']]
y = df['tiempo_viaje_min']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.2, random_state=42)
modelo = Pipeline([('imputador', SimpleImputer(strategy='median')),
                   ('escalador', StandardScaler()),
                   ('regresion', LinearRegression())])
modelo.fit(X_train,y_train)
pred = modelo.predict(X_test)
print('Entrenamiento:',len(X_train),'Prueba:',len(X_test))

## 3. Modelo seleccionado: regresión lineal múltiple
La variable objetivo (minutos) es cuantitativa continua. La regresión lineal es un **modelo base interpretable** y permite estimar la relación entre predictores y respuesta. Su adecuación real requeriría datos de operación, análisis de residuos y comparación con otros modelos.

In [ ]:
mae = mean_absolute_error(y_test,pred)
rmse = np.sqrt(mean_squared_error(y_test,pred))
r2 = r2_score(y_test,pred)
print(f'MAE: {mae:.2f} min | RMSE: {rmse:.2f} min | R²: {r2:.3f}')
coefs = pd.Series(modelo.named_steps['regresion'].coef_, index=X.columns)
print('Coeficientes con variables estandarizadas:'); display(coefs.to_frame('coeficiente'))

## 4. Visualización e interpretación
En el gráfico, cuanto más cerca estén los puntos de la diagonal, mejor coincide la predicción con los valores reservados para prueba. El buen rendimiento esperado refleja que los datos se generaron mediante una relación aproximadamente lineal; **no demuestra rendimiento con datos reales**.

In [ ]:
fig, ax = plt.subplots(figsize=(7,5))
ax.scatter(y_test,pred,alpha=.7)
lo = min(y_test.min(),pred.min()); hi=max(y_test.max(),pred.max())
ax.plot([lo,hi],[lo,hi],linestyle='--',color='black')
ax.set(xlabel='Tiempo real simulado (min)', ylabel='Tiempo predicho (min)',title=f'Predicción vs. simulación | MAE={mae:.2f} min, R²={r2:.2f}')
plt.tight_layout(); plt.show()
fig, ax = plt.subplots(figsize=(7,4))
ax.scatter(pred,y_test-pred,alpha=.7)
ax.axhline(0,linestyle='--',color='black')
ax.set(xlabel='Predicción (min)',ylabel='Residuo: real - predicho (min)',title='Residuos en test')
plt.tight_layout(); plt.show()

## 5. Predicción ilustrativa y conclusión
Se simula un trayecto de 15 km con 120 t y ley 0.9 % para ejemplificar el uso de la regresión. **No equivale a una estimación certificada para el frente D**, porque el conjunto completo es sintético.

In [ ]:
ejemplo = pd.DataFrame([{'distancia_km':15,'carga_ton':120,'ley_cobre_pct':.9}])
print(f'Tiempo de viaje estimado para condiciones simuladas: {modelo.predict(ejemplo)[0]:.1f} min')